# RAG simples com Ollama e Huggung Face (nomic-embed-text)

Este notebook lê uma uma pasta DATA_DIR com PDFs, faz chunking, gera embeddings, com `nomic-embed-text` no Ollama ou Hugging Face, e insere no ChromaDB. 

Recupera exemplos de chunks mais similares para uma pergunta com teste e armazena parte dos dados para verificação e auditoria em um JSON.

**Pré-requisito:** Ollama em execução em `http://localhost:11434` ou conexão com o Hugging Face e modelos baixados (`nomic-embed-text`).

## 1) Instalar dependências


In [ ]:
!pip -q install requests numpy chromadb pypdf tqdm torch transformers einops

## 2) Configurações


In [ ]:
import os
import json
import requests
import re
import uuid
import chromadb
import torch
from tqdm import tqdm
from pathlib import Path
from pypdf import PdfReader
import torch.nn.functional as F
from chromadb.config import Settings
from dotenv import load_dotenv, find_dotenv
from transformers import AutoTokenizer, AutoModel

# --------------------------------------
# Carrega variáveis do .env
# --------------------------------------
load_dotenv()

# =========================
# Configuração
# =========================
NOMIC_MODEL = "nomic-ai/nomic-embed-text-v1.5"

# =========================
# Load tokenizer
# =========================
_tokenizer = AutoTokenizer.from_pretrained(NOMIC_MODEL, trust_remote_code=True)
_model = AutoModel.from_pretrained(
    NOMIC_MODEL,
    trust_remote_code=True,
    safe_serialization=True
)
_model.eval()

BATCH = int(os.getenv("BATCH", 64))

# garante expansão do ~ no Linux
DATA_DIR = Path(os.getenv("DATA_DIR") + f"/diario_oficial_menor").expanduser()

DEVICE = os.getenv("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = os.getenv("PROJECT_ROOT")

COLLECTION_NAME = os.getenv("COLLECTION_NAME", "base_pdfs")

# Chunking
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", 2048))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", 150))
MIN_CHUNK_LEN = int(os.getenv("MIN_CHUNK_LEN", 200))

# Paths derivados
CHROMADB_PATH = f"chromadb_chunk_size_{CHUNK_SIZE}_overlap_{CHUNK_OVERLAP}"
INDEX_PATH = Path(f"{PROJECT_ROOT}/{CHROMADB_PATH}").expanduser()

AUDIT_PATH = Path(f"{PROJECT_ROOT}/chunks_audit_{CHUNK_SIZE}.json").expanduser()

# Ollama
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")

# Retrieval
TOP_K = int(os.getenv("TOP_K", 5))

<All keys matched successfully>


### 2.1) Verifica configurações

In [2]:
print("Diretório atual:", os.getcwd())
dotenv_path = find_dotenv()
print("Arquivo .env encontrado:", dotenv_path)

vars_env = [
    "DATA_DIR",
    "PROJECT_ROOT",
    "COLLECTION_NAME",
    "CHUNK_SIZE",
    "CHUNK_OVERLAP",
    "MIN_CHUNK_LEN",
    "OLLAMA_BASE_URL",
    "EMBED_MODEL",
    "TOP_K",
    "BATCH"
]

print("\n===== TESTE DAS VARIÁVEIS DE AMBIENTE =====\n")

for v in vars_env:
    value = os.getenv(v)

    if value is None:
        print(f"{v:20} -> ❌ NÃO CARREGADA (None)")
    else:
        print(f"{v:20} -> ✅ {value}")

print(f"INDEX_PATH".ljust(20) + f" -> ✅ {INDEX_PATH}")
print(f"AUDIT_PATH".ljust(20) + f" -> ✅ {AUDIT_PATH}")
print(f"DEVICE".ljust(20) + f" -> ✅ {DEVICE}")

Diretório atual: /home/raulcarvalho/RAG-simples
Arquivo .env encontrado: /home/raulcarvalho/RAG-simples/.env

===== TESTE DAS VARIÁVEIS DE AMBIENTE =====

DATA_DIR             -> ✅ ~/Experimentos/DiarioOficial-Contratos-BR-GT/
PROJECT_ROOT         -> ✅ ~/Experimentos/Resultados
COLLECTION_NAME      -> ✅ base_pdfs
CHUNK_SIZE           -> ✅ 512
CHUNK_OVERLAP        -> ✅ 150
MIN_CHUNK_LEN        -> ✅ 200
OLLAMA_BASE_URL      -> ✅ http://localhost:11434
EMBED_MODEL          -> ✅ nomic-embed-text
TOP_K                -> ✅ 2
BATCH                -> ✅ 64
INDEX_PATH           -> ✅ /home/raulcarvalho/Experimentos/Resultados/chromadb_chunk_size_512_overlap_150
AUDIT_PATH           -> ✅ /home/raulcarvalho/Experimentos/Resultados/chunks_audit_512.json
DEVICE               -> ✅ cuda


## 3) Funções de chamada embeddings por Ollama ou Hugging Face


In [1]:
def ollama_embed(text: str, model: str = EMBED_MODEL) -> list[float]:
    """Gera embedding via Ollama /api/embeddings."""
    url = f"{OLLAMA_BASE_URL}/api/embeddings"
    payload = {"model": model, "prompt": text}
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    data = r.json()
    return data["embedding"]

# =========================
# Pooling
# =========================
def _mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(
        input_mask_expanded.sum(dim=1), min=1e-9
    )


# =========================
# Hugging Face Embedding
# =========================
def hf_nomic_embed(
    text: str,
    prefix: str = "search_document:",
    device: str = DEVICE
) -> list[float]:
    """
    Gera embedding usando Nomic local do Hugging Face.

    Parâmetros:
      - text: texto de entrada
      - prefix:
          * 'search_document:' para indexação
          * 'search_query:' para consulta
      - device: 'cpu', 'cuda' ou 'auto'
    """

    if not text:
        return []

    # move o modelo para o device desejado
    model = _model.to(device)
    model.eval()

    # prefixo do modelo usado no nomic-embedding
    text = f"{prefix} {text}"

    encoded_input = _tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # move inputs para o mesmo device
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}

    with torch.no_grad():
        model_output = model(**encoded_input)

    embeddings = _mean_pooling(model_output, encoded_input["attention_mask"])

    # pipeline Nomic
    embeddings = F.layer_norm(embeddings, normalized_shape=(embeddings.shape[1],))
    embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings[0].detach().cpu().tolist()

NameError: name 'EMBED_MODEL' is not defined

## 4) Função para gerar chunks


In [ ]:
def clean_text(s: str) -> str:
    if not s:
        return ""
    s = re.sub(r"\s+", " ", s).strip()
    return s


def chunk_text(
    text: str,
    tokenizer,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP
):

    # chunking por tokens usando tokenizer (ex: Nomic).

    text = clean_text(text)
    if not text:
        return []

    # Tokeniza sem truncar
    tokens = tokenizer.encode(text, add_special_tokens=False)

    if len(tokens) == 0:
        return []

    chunks = []
    start = 0

    while start < len(tokens):
        end = min(len(tokens), start + chunk_size)

        chunk_tokens = tokens[start:end]

        # reconstrói texto do chunk
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True).strip()

        # usa regra de tamanho mínimo
        if len(chunk_text) >= MIN_CHUNK_LEN:
            chunks.append(chunk_text)

        if end == len(tokens):
            break

        start = max(0, end - overlap)

    return chunks

## Gerar embeddings e chunking

In [5]:
pdf_paths = sorted([p for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".pdf"])
print(f"📄 PDFs encontrados: {len(pdf_paths)} em {DATA_DIR}")

docs = []

for pdf_path in tqdm(pdf_paths, desc="Lendo PDFs"):
    try:
        reader = PdfReader(str(pdf_path))
    except Exception as e:
        print(f"⚠️ Falha ao abrir {pdf_path.name}: {e}")
        continue

    for page_idx, page in enumerate(reader.pages, start=1):
        try:
            page_text = page.extract_text() or ""
        except Exception:
            page_text = ""

        page_chunks = chunk_text(
            text=page_text,
            tokenizer=_tokenizer,
            chunk_size=CHUNK_SIZE,
            overlap=CHUNK_OVERLAP
        )

        if not page_chunks:
            continue

        page_embeddings = [
            hf_nomic_embed(chunk, prefix="search_document:")
            for chunk in page_chunks
        ]

        for chunk_idx, (chunk, emb) in enumerate(zip(page_chunks, page_embeddings), start=1):
            docs.append({
                "id": str(uuid.uuid4()),
                "text": chunk,
                "embedding": emb, 
                "metadata": {
                    "source_pdf": pdf_path.name,
                    "page": page_idx,
                    "chunk": chunk_idx,
                    "chunk_tokens": len(_tokenizer.encode(chunk, add_special_tokens=False))
                }
            })

print(f"✅ Total de chunks gerados: {len(docs)}")

📄 PDFs encontrados: 36 em /home/raulcarvalho/Experimentos/DiarioOficial-Contratos-BR-GT/diario_oficial_menor


Lendo PDFs: 100%|██████████| 36/36 [16:27<00:00, 27.44s/it]

✅ Total de chunks gerados: 26998


## 5) Criar/atualizar índice no ChromaDB (persistente)


In [6]:
INDEX_DIR = Path(INDEX_PATH)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

settings = Settings(
    _env_file=None,               # << impede tentar ler ".env"
    anonymized_telemetry=False
)

client = chromadb.PersistentClient(path=str(INDEX_DIR), settings=settings)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print("📦 Coleção:", COLLECTION_NAME)
print("📁 Persistência em:", INDEX_DIR)
print("🔢 Itens atuais na coleção:", collection.count())


📦 Coleção: base_pdfs
📁 Persistência em: /home/raulcarvalho/Experimentos/Resultados/chromadb_chunk_size_512_overlap_150
🔢 Itens atuais na coleção: 0


## 6) Inserir (com batch simples) no ChromaDB


In [7]:
if not docs:
    raise RuntimeError("Não há chunks (docs) para inserir. Verifique DATA_DIR, o chunking e a geração de embeddings.")

for i in tqdm(range(0, len(docs), BATCH), desc="Ingerindo no Chroma"):
    batch = docs[i:i+BATCH]

    ids = [d["id"] for d in batch]
    texts = [d["text"] for d in batch]
    metas = [d["metadata"] for d in batch]
    embeds = [d["embedding"] for d in batch]   # <- usa embeddings já gerados

    collection.add(
        ids=ids,
        documents=texts,
        metadatas=metas,
        embeddings=embeds,
    )

print(f"✅ Ingestão concluída. Total na coleção: {collection.count()}")

Ingerindo no Chroma:  10%|█         | 44/422 [00:33<05:39,  1.11it/s]

Ingerindo no Chroma: 100%|██████████| 422/422 [09:32<00:00,  1.36s/it]

✅ Ingestão concluída. Total na coleção: 26998


## 7) Consulta rápida (recuperação top-k)


In [8]:
def chroma_retrieve(query: str, top_k: int = TOP_K):
    q_emb = hf_nomic_embed(query)
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    # organiza
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"distance": float(dist), "metadata": meta, "text": doc})
    return hits

query = "Qual é o valor total do contrato de pneus?"
hits = chroma_retrieve(query, top_k=5)

for i, h in enumerate(hits, start=1):
    m = h["metadata"]
    print(f"\n=== Hit {i} | dist={h['distance']:.4f} | {m['source_pdf']} | pág {m['page']} | chunk {m['chunk']} ===")
    print(h["text"][:900], "...")



=== Hit 1 | dist=0.2276 | DODF 236 20-12-2021 INTEGRA.pdf | pág 57 | chunk 4 ===
##o. valor : o valor total do contrato e de r $ 53. 328, 00 ( cinquenta e tres mil, trezentos e vinte oito e sessenta centavos ), o empenho e de r $ 53. 328, 00 ( cinquenta e tres mil, trezentos e vinte oito e sessenta centavos ), conforme nota de empenho nº 2021ne00670 emitida em 16 / 12 / 2021, sob o evento400091, na modalidade global, unidade orcamentaria : 14101. programa de trabalho : 20. 306. 6201. 4115. 0007, natureza da despesa 339032. fonte de recurso : 100 id. uso 6 prazo de vigencia : o contrato tera vigencia desde a sua assinatura por ate 12 ( doze ) meses. data de assinatura : 17 / 12 / 2021. signatarios : pela seagri / df : candido teles de araujo, na qualidade de secretario de estado. pela contratada : luciano andrade de carvalho, na qualidade representante legal. extrato do contrato para aquisicao de bens pelo distrito federal nº 45 / 2021 processo : 00070 - 00004017 / 20 ...

=== Hit 2 | 

## 8) Salvar chunks/metadata em JSON para auditoria


In [9]:
with open(AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        [{"id": d["id"], **d["metadata"], "text_preview": d["text"]} for d in docs],
        f, ensure_ascii=False, indent=2
    )

print("📝 Auditoria salva em:", AUDIT_PATH)


📝 Auditoria salva em: /home/raulcarvalho/Experimentos/Resultados/chunks_audit_512.json


In [10]:
# Recuperar o primeiro item da coleção

data = collection.get(
    limit=1,
    include=["documents", "metadatas"]
)

print("ID:", data["ids"][0])
print("\nTexto do chunk:\n")
print(data["documents"][0][:500])

print("\nMetadata:")
print(data["metadatas"][0])

ID: 5f75ccf4-f166-4089-a5d0-9201b648a2c9

Texto do chunk:

ano li edicao nº 11 brasilia - df, segunda - feira, 17 de janeiro de 2022 sumario secao i pag. secao ii pag. secao iii pag. poder executivo......................................................... 1 21 secretaria de estado de governo............................... 2 21 35 secretaria de estado de economia............................. 4 22 35 secretaria de estado de saude................................... 7 23 37 secretaria de estado de educacao............................. 7 26 43 secretaria de 

Metadata:
{'chunk_tokens': 512, 'chunk': 1, 'source_pdf': 'DODF 011 17-01-2022 INTEGRA.pdf', 'page': 1}


In [11]:
# recuperar embedding do primeiro item

first_id = data["ids"][0]

emb = collection.get(
    ids=[first_id],
    include=["embeddings"]
)["embeddings"][0]

print("\nDimensão do embedding:", len(emb))
print("Primeiros valores:", emb[:10])


Dimensão do embedding: 768
Primeiros valores: [ 0.0056687   0.06270751 -0.16680142  0.02599802  0.04690583 -0.04561435
  0.00140407 -0.00834492 -0.00848214  0.01938872]


In [12]:
del collection
del client